# S3 — Éval Oolel-v0.1 vs extraction directe (D1)

Tranche D1 (génération, [JOURNAL_2026-07_08](../docs/Journal-Decisions/JOURNAL_2026-07_08.md)) : `soynade-research/Oolel-v0.1` (nativement wolof, 7,6 B, Qwen2.5, Apache 2.0) apporte-t-il une valeur suffisante sur l'extraction directe (méthode de repli validée en S2-J5) pour devenir la baseline de génération du rapport ?

**Protocole** : fp16 sur 2×T4 (Kaggle), pas le 4-bit prévu au plan initial — le checkpoint est en fp32 sur disque (~30 Go) et la quantification 4-bit n'a pas pu être posée dans ce runtime. Ce run mesure donc le **plafond qualité** du modèle, pas la variante de déploiement.

11 questions (10 in-corpus + 1 hors-corpus délibérée — prix/vente du lait, non couvert par les fiches) tirées de [`eval_set.json`](../eval_set.json), jeu figé par le retrieval hybride (`08_indexation_wolof.ipynb`). Comparaison par question : réponse Oolel vs `passages[0]` brut.

**Notation manuelle, hors notebook** — grille détaillée en fin de notebook, travail en cours.

## Env — dépendances + auth HF

Installe `transformers`/`accelerate` (versions Kaggle par défaut pas toujours compatibles avec `Oolel-v0.1`). Connexion HF (`HF_TOKEN` via Kaggle Secrets) nécessaire pour télécharger le checkpoint fp32 (~30 Go).

In [ ]:
#env
!pip -q install -U transformers accelerate
import gc, time, json
import torch, pandas as pd
from transformers import AutoModelForCausalLM, AutoTokenizer

In [ ]:
from kaggle_secrets import UserSecretsClient
from huggingface_hub import login

user_secrets = UserSecretsClient()
secret_value_0 = user_secrets.get_secret("HF_TOKEN")
login(secret_value_0)

## Config

Un seul modèle testé (`Oolel-v0.1`) — dict `MODELS` prévu pour en ajouter d'autres sans toucher au harnais. Génération greedy (`do_sample=False`) pour reproductibilité. Sortie : `/kaggle/working/eval_llm_wolof.csv`.

In [ ]:
#config
MODELS = {
    "oolel_v01": "soynade-research/Oolel-v0.1",   # 7,6B, qwen2, Apache 2.0
}
GEN = dict(max_new_tokens=384, do_sample=False)   # greedy, reproductible
OUT_CSV = "/kaggle/working/eval_llm_wolof.csv"

## Données de test + prompt

Charge `eval_set.json` depuis le dataset Kaggle attaché — jeu figé, retrieval non réexécuté ici. Consigne système en wolof strict : répondre en wolof uniquement, s'appuyer sur le contexte fourni, ne rien inventer si l'info manque (couvre le cas hors-corpus). `build_messages` assemble contexte + question au format chat.

In [ ]:
#tes parties
from pathlib import Path

# chargement du jeu figé depuis le dataset Kaggle ---
EVAL_PATH = Path("/kaggle/input/datasets/serignedanfall/noofar-eval-set/eval_set.json")
EVAL = json.loads(EVAL_PATH.read_text(encoding="utf-8"))
print(f"{len(EVAL)} questions chargées")

# consigne système wolof ---
SYSTEM_WO = "Yaa ngi jàppale sàmmkat yi. Tontu ci wolof rekk. Sukkandiku ci leeral yiñ la jox; sudee leeral yi nekkul ci xibaar bi, wax ko—bul sos dara. Wax ci anam wu leer te yomb, ci anam wu sàmmkat bi mëna dégg."

# construction des messages (contexte + question) ---
def build_messages(question, passages):
    context = "\n\n".join(passages)
    return [
        {"role": "system", "content": SYSTEM_WO},
        {"role": "user",   "content": f"Xibaar bi:\n{context}\n\nLaaj bi: {question}"},
    ]

## Harnais

`load()` charge le modèle en fp16 (`dtype=torch.float16` — pas de bf16 sur T4), `device_map="auto"` répartit les couches sur les 2 GPU. `generate()` applique le chat template et ne décode que les tokens générés. `free()` vide le cache CUDA entre modèles.

In [ ]:
# Harnais

def free():
    gc.collect()
    for i in range(torch.cuda.device_count()):
        with torch.cuda.device(i): torch.cuda.empty_cache()

def load(model_id):
    tok = AutoTokenizer.from_pretrained(model_id)
    model = AutoModelForCausalLM.from_pretrained(
        model_id, dtype=torch.float16, device_map="auto",  # T4 = pas de bf16
    ).eval()
    print(model_id, "->", model.hf_device_map)
    return tok, model

def generate(tok, model, question, passages):
    encoded = tok.apply_chat_template(
        build_messages(question, passages),
        add_generation_prompt=True,
        return_tensors="pt",
    )
    # encoded peut être un BatchEncoding ou un tenseur selon la version transformers
    if hasattr(encoded, "input_ids"):
        ids = encoded.input_ids.to("cuda:0")
    else:
        ids = encoded.to("cuda:0")
    t0 = time.time()
    with torch.no_grad():
        out = model.generate(ids, **GEN)
    dt = time.time() - t0
    return tok.decode(out[0, ids.shape[-1]:], skip_special_tokens=True), round(dt, 2)

## Run + export

Boucle sur `MODELS` (un seul ici) × les 11 questions : réponse + latence par question. Colonnes de jugement ajoutées vides. Export vers `OUT_CSV`.

In [ ]:
#run + export
rows = []
for it in EVAL:
    rows.append({
        "question":       it["question"],
        "intent_attendu": it["intent_attendu"],
        "source_top":     it["sources"][0],
        "extraction":     it["passages"][0],
    })

for name, model_id in MODELS.items():
    try:
        tok, model = load(model_id)
    except Exception as e:
        print(f"[skip] {name}: {e}")
        continue
    for r, it in zip(rows, EVAL):
        txt, dt = generate(tok, model, it["question"], it["passages"])
        r[name] = txt
        r[f"{name}_sec"] = dt   # latence de banc 2xT4, PAS déploiement
    del model, tok
    free()

df = pd.DataFrame(rows)

# colonnes de jugement (binaire 0/1) — à remplir à la main après le run
for col in ["wolof_ok", "fidele", "oral", "mieux_que_extraction", "remarques"]:
    df[col] = ""

df.to_csv(OUT_CSV, index=False)
print("saved:", OUT_CSV)
df

## Grille de notation (manuelle, hors notebook)

| Colonne | Sens |
|---|---|
| `wolof_ok` | Réponse en wolof correct — pas de bascule FR/EN, pas de charabia |
| `fidele` | Fidèle au contexte fourni, aucune information inventée |
| `oral` | Formulation adaptée à un éleveur qui écoute, pas qui lit |
| `mieux_que_extraction` | Réponse Oolel préférée à `passages[0]` brut |
| `remarques` | Texte libre |